# 16 — PSO Estimator Stability / Uncertainty Quantification

This notebook quantifies **estimator/seed uncertainty** at fixed data.

**Tasks:**
- **Step 0 (gate):** confirm that seeding `np.random.seed(k)` before constructing a fresh `GlobalBestPSO`
  makes a single CBG's optimisation exactly reproducible.
- **Steps 1–2:** draw a 300-CBG stratified sample (borough × income quintile, seed 123) from the pooled
  5,502-CBG analytical sample, then run PSO **10 times** (seeds 0–9) per CBG per year 2018–2021 —
  **12,000 runs** — recorded in `all_seed_runs.csv`.
- **Steps 3–7:** per-(cbg,year) coefficient of variation across seeds; population-level %-change dispersion
  across the 10 seeds; cross-CBG pairwise seed correlations; directional (sign) agreement; and a fit
  comparison against the original full-sample runs.

Bounds here are the feature-scale ceiling `[1,28]` used directly as the exponent search bound (as specified
for this stability study). **Reproducibility / safety:** every run is seeded; `WRITE_OUTPUTS=False` so the
notebook recomputes in memory and verifies against the saved `all_seed_runs.csv` / `stage_results.txt`
without modifying them.


In [ ]:
import os, numpy as np, pandas as pd, logging
logging.getLogger('pyswarms').setLevel(logging.ERROR)
from pyswarms.single.global_best import GlobalBestPSO
from scipy.stats import pearsonr
pd.set_option('display.width',160); pd.set_option('display.max_columns',None)

REPO='/Users/mohsenbahrami/Desktop/frontiers_repo'
DATA=os.path.join(REPO,'data/model_inputs')
def find_dir(*cands):   # output dir renamed between item*_ and bare variants; auto-detect
    for c in cands:
        p=os.path.join(REPO,'outputs',c)
        if os.path.isdir(p): return p
    raise FileNotFoundError(f'none of {cands} found under outputs/')
OUT=find_dir('item5_pso_stability','pso_stability')
YEARS=[2018,2019,2020,2021]; SEEDS=list(range(10))
N_SAMPLE=300; SAMPLE_SEED=123; NORM=28; PSO_RANGE=28
WRITE_OUTPUTS=False
PARAM=['H_Area_of_store','R_Percentage_of_Visits_by_brand','J_POI_count_where_store_is',
       'K_POI_diversity_where_store_is','L_Demographic_similarity','G_Distance_between_cbg_and_store']
PRETTY={'H_Area_of_store':'Store area','R_Percentage_of_Visits_by_brand':'Chain loyalty',
        'J_POI_count_where_store_is':'POI count','K_POI_diversity_where_store_is':'POI diversity',
        'L_Demographic_similarity':'Demographic similarity','G_Distance_between_cbg_and_store':'Distance'}
BOROUGH={5:'Bronx',47:'Brooklyn',61:'Manhattan',81:'Queens',85:'Staten Island'}
def borough_of(c): return BOROUGH.get((c//10**7)%1000,'Unknown')
def convert(x,lo,hi): return ((x-lo)/(hi-lo))*(NORM-1)+1 if hi!=lo else pd.Series(1.0,index=x.index)
OPTIONS={'c1':1.5,'c2':1.5,'w':0.9}; BOUNDS=(np.array([1.]*6),np.array([float(PSO_RANGE)]*6))
PASS=[]
def check(name,ok,detail=''):
    PASS.append((name,ok,detail)); print(('PASS ' if ok else '*** FAIL *** ')+name+('  '+detail if detail else ''))

## Step 0 — Seeding reproducibility gate

In [ ]:
# same seed + fresh optimiser on the same CBG must give identical output
cols=['A_cbg','B_store','C_Percentage_of_Visits_2019','H_Area_of_store','R_Percentage_of_Visits_by_brand_2019',
      'J_POI_count_where_store_is','K_POI_diversity_where_store_is','L_Demographic_similarity','G_Distance_between_cbg_and_store']
t=pd.read_csv(os.path.join(DATA,'table_2019.csv'),usecols=cols).rename(
    columns={'R_Percentage_of_Visits_by_brand_2019':'R_Percentage_of_Visits_by_brand'})
for a in PARAM: t[a]=convert(t[a],float(t[a].min()),float(t[a].max()))
g0=t[t['A_cbg']==sorted(t['A_cbg'].unique())[0]]
def make_opt(g):
    H,R,J,K,L,G=[g[c].values.astype(float) for c in PARAM]
    actual=g['C_Percentage_of_Visits_2019'].values.astype(float) if 'C_Percentage_of_Visits_2019' in g else g['actual'].values.astype(float)
    am=actual.mean(); asd=actual.std()
    def opt(X):
        out=np.empty(X.shape[0])
        for i in range(X.shape[0]):
            p=X[i]; a=(H**p[0])*(R**p[1])*(J**p[2])*(K**p[3])*(L**p[4])/(G**p[5])
            s=a.sum()
            if s!=0: a=a/s
            bsd=a.std()
            out[i]=1.0 if (asd==0 or bsd==0) else 1.0-np.mean((actual-am)*(a-a.mean()))/(asd*bsd)
        return out
    return opt
def once(g,seed):
    np.random.seed(seed)
    o=GlobalBestPSO(n_particles=20,dimensions=6,options=OPTIONS,bounds=BOUNDS)
    return o.optimize(make_opt(g),iters=10,verbose=False)
c1,v1=once(g0,0); c2,v2=once(g0,0)
check('STEP 0 seeding reproducible (identical)', (c1==c2) and np.array_equal(v1,v2))
assert (c1==c2) and np.array_equal(v1,v2), 'STEP 0 FAILED — not reproducible'
del t

## Step 1 — Stratified 300-CBG sample (seed 123)

In [ ]:
inc=pd.read_csv(os.path.join(DATA,'table_2018.csv'),usecols=['A_cbg','M_Median_Income_in_this_cbg'])
income=inc.groupby('A_cbg')['M_Median_Income_in_this_cbg'].first()
pool=pd.DataFrame({'A_cbg':income.index})
pool['borough']=pool['A_cbg'].map(borough_of); pool['income']=pool['A_cbg'].map(income)
pool['quintile']=pd.qcut(pool['income'],5,labels=['Q1','Q2','Q3','Q4','Q5']).astype(str)
n_pool=len(pool); picks=[]
for (b,q),g in pool.groupby(['borough','quintile']):
    n_s=min(int(round(N_SAMPLE*len(g)/n_pool)),len(g))
    if n_s>0: picks.append(g.sample(n=n_s,random_state=SAMPLE_SEED))
samp=pd.concat(picks).reset_index(drop=True)
sample_cbgs=set(samp['A_cbg'])
print('pool=%d sampled=%d'%(n_pool,len(samp)))
for dim in ['borough','quintile']:
    pc=pool[dim].value_counts().sort_index(); sc=samp[dim].value_counts().sort_index()
    print('\nby',dim); print(pd.DataFrame({'pool_%':(pc/n_pool*100).round(1),'sample_%':(sc/len(samp)*100).round(1)}).to_string())
# verify against saved all_seed_runs.csv CBG set
saved=pd.read_csv(os.path.join(OUT,'all_seed_runs.csv'))
check('sampled CBG set matches saved all_seed_runs', set(samp['A_cbg'])==set(saved['cbg'].unique()),
      f'sampled {len(sample_cbgs)} vs saved {saved["cbg"].nunique()}')

## Step 2 — 12,000 PSO runs (300 CBGs × 4 years × 10 seeds)

In [ ]:
rows=[]
for year in YEARS:
    cols=['A_cbg','B_store',f'C_Percentage_of_Visits_{year}','H_Area_of_store',
          f'R_Percentage_of_Visits_by_brand_{year}','J_POI_count_where_store_is',
          'K_POI_diversity_where_store_is','L_Demographic_similarity','G_Distance_between_cbg_and_store']
    t=pd.read_csv(os.path.join(DATA,f'table_{year}.csv'),usecols=cols).rename(
        columns={f'R_Percentage_of_Visits_by_brand_{year}':'R_Percentage_of_Visits_by_brand'})
    for a in PARAM: t[a]=convert(t[a],float(t[a].min()),float(t[a].max()))
    t['actual']=t[f'C_Percentage_of_Visits_{year}']
    t=t[t['A_cbg'].isin(sample_cbgs)]
    groups=dict(tuple(t.groupby('A_cbg')))
    for c in sorted(groups):
        opt=make_opt(groups[c])
        for k in SEEDS:
            np.random.seed(k)
            o=GlobalBestPSO(n_particles=20,dimensions=6,options=OPTIONS,bounds=BOUNDS)
            cost,v=o.optimize(opt,iters=10,verbose=False)
            rows.append([c,year,k,cost]+list(v))
    print(f'year {year} done')
    del t
R=pd.DataFrame(rows,columns=['cbg','year','seed','cost']+PARAM)
print('total runs:',len(R))
check('produced 12,000 runs', len(R)==12000, f'got {len(R)}')

In [ ]:
# --- VERIFY fresh runs reproduce saved all_seed_runs.csv (allclose on aligned keys) ---
sv=pd.read_csv(os.path.join(OUT,'all_seed_runs.csv')).sort_values(['cbg','year','seed']).reset_index(drop=True)
fr=R.sort_values(['cbg','year','seed']).reset_index(drop=True)
keys_match=(sv[['cbg','year','seed']].values==fr[['cbg','year','seed']].values).all()
vals_match=np.allclose(sv[['cost']+PARAM].values, fr[['cost']+PARAM].values, rtol=1e-9, atol=1e-9)
check('fresh 12k runs reproduce saved all_seed_runs.csv', bool(keys_match and vals_match))

## Step 3 — Per-(cbg,year) stability: median CV across the 10 seeds

In [ ]:
def pct(a,b): return (b-a)/a*100
grp=R.groupby(['cbg','year'])
cv=(grp[PARAM].std()/grp[PARAM].mean().abs()).reset_index()
cvtab=pd.DataFrame({PRETTY[c]:cv.groupby('year')[c].median() for c in PARAM})
print(cvtab.round(4).to_string())
# expected median CV from saved stage_results.txt
EXP_CV={2018:[0.4757,0.4782,0.5286,0.5556,0.5069,0.2706],2019:[0.4836,0.4829,0.5321,0.5394,0.5172,0.2637],
        2020:[0.3742,0.3154,0.5069,0.5057,0.5099,0.3465],2021:[0.3487,0.3124,0.4903,0.5083,0.5180,0.4508]}
okcv=all(abs(round(cvtab.loc[y,PRETTY[c]],4)-EXP_CV[y][i])<=0.0001 for y in YEARS for i,c in enumerate(PARAM))
check('median CV matches saved report', okcv)

## Step 4 — Population-level %-change dispersion across the 10 seeds

In [ ]:
seedmeans={k:{y:R[(R.seed==k)&(R.year==y)][PARAM].mean() for y in YEARS} for k in SEEDS}
EXP4={  # (pair, param): (mean,std,min,max) from saved report
 ((2019,2020),'Store area'):(68.15,14.04,42.86,88.07),((2019,2020),'Chain loyalty'):(29.86,11.82,7.29,46.86),
 ((2019,2020),'POI count'):(24.87,10.20,8.63,39.88),((2019,2020),'POI diversity'):(11.18,6.82,-1.16,18.25),
 ((2019,2020),'Demographic similarity'):(2.77,6.28,-4.12,16.41),((2019,2020),'Distance'):(-23.61,3.49,-28.72,-18.90),
 ((2019,2021),'Store area'):(78.44,17.98,46.28,109.99),((2019,2021),'Chain loyalty'):(35.19,12.43,14.55,58.66),
 ((2019,2021),'POI count'):(52.60,12.67,29.72,69.34),((2019,2021),'POI diversity'):(12.36,11.71,-7.19,27.16),
 ((2019,2021),'Demographic similarity'):(-0.36,9.53,-10.72,24.08),((2019,2021),'Distance'):(-35.18,3.98,-42.86,-29.61)}
ok4=True
for pair in [(2019,2020),(2019,2021)]:
    print(f'\n%change {pair[0]}->{pair[1]} across 10 seeds:')
    print(f"  {'Parameter':24s} {'mean':>8s} {'std':>8s} {'min':>8s} {'max':>8s}")
    for c in PARAM:
        v=np.array([pct(seedmeans[k][pair[0]][c],seedmeans[k][pair[1]][c]) for k in SEEDS])
        got=(round(v.mean(),2),round(v.std(),2),round(v.min(),2),round(v.max(),2))
        exp=EXP4[(pair,PRETTY[c])]
        if any(abs(got[j]-exp[j])>0.01 for j in range(4)): ok4=False; print('   MISMATCH',c,got,'vs',exp)
        print(f"  {PRETTY[c]:24s} {got[0]:>7.2f}% {got[1]:>7.2f}% {got[2]:>7.2f}% {got[3]:>7.2f}%")
check('population %-change (mean/std/min/max) matches report', ok4)

## Step 5 — Cross-CBG pairwise Pearson r between the 10 seeds

In [ ]:
EXP5={(2018,'Store area'):(0.4940,0.3698),(2019,'Chain loyalty'):(0.3958,0.2941),
      (2020,'POI count'):(0.5968,0.5166),(2021,'POI count'):(0.7100,0.6363),(2020,'Distance'):(0.5868,0.4742)}
ok5=True
print(f"  {'year':5s} {'Parameter':24s} {'mean_r':>8s} {'min_r':>8s}")
for year in YEARS:
    sy=R[R.year==year]
    for c in PARAM:
        piv=sy.pivot_table(index='cbg',columns='seed',values=c).dropna()
        rs=[pearsonr(piv[i],piv[j])[0] for i in SEEDS for j in SEEDS if j>i and piv[i].std()>0 and piv[j].std()>0]
        rs=np.array(rs); mr,mn=round(rs.mean(),4),round(rs.min(),4)
        if (year,PRETTY[c]) in EXP5:
            e=EXP5[(year,PRETTY[c])]
            if abs(mr-e[0])>0.001 or abs(mn-e[1])>0.001: ok5=False; print('   MISMATCH',year,c,(mr,mn),'vs',e)
        print(f"  {year:<5d} {PRETTY[c]:24s} {mr:>8.4f} {mn:>8.4f}")
check('spot-checked pairwise seed correlations match report', ok5)

## Step 6 — Directional (sign) agreement across seeds

In [ ]:
EXP6={((2019,2020),'Store area'):10,((2019,2020),'Chain loyalty'):10,((2019,2020),'POI count'):10,
      ((2019,2020),'POI diversity'):8,((2019,2020),'Demographic similarity'):5,((2019,2020),'Distance'):10,
      ((2019,2021),'Store area'):10,((2019,2021),'Chain loyalty'):10,((2019,2021),'POI count'):10,
      ((2019,2021),'POI diversity'):8,((2019,2021),'Demographic similarity'):6,((2019,2021),'Distance'):10}
ok6=True
for pair in [(2019,2020),(2019,2021)]:
    print(f'\n{pair[0]}->{pair[1]} majority-sign agreement:')
    for c in PARAM:
        s=np.array([np.sign(pct(seedmeans[k][pair[0]][c],seedmeans[k][pair[1]][c])) for k in SEEDS])
        maj=max((s>0).sum(),(s<0).sum())
        if maj!=EXP6[(pair,PRETTY[c])]: ok6=False; print('   MISMATCH',c,maj,'vs',EXP6[(pair,PRETTY[c])])
        print(f"  {PRETTY[c]:24s} {maj}/10")
check('directional agreement matches report', ok6)

## Step 7 — Fit comparison vs original full-sample runs

In [ ]:
cm=R.groupby(['cbg','year'])['cost'].mean().reset_index()
EXP7={2018:0.7231,2019:0.7440,2020:0.1712,2021:0.1496}
ok7=True
print(f"  {'year':5s} {'stability median fit':>20s} {'orig mean fit':>14s}")
for year in YEARS:
    medfit=round(1-cm[cm.year==year]['cost'].median(),4)
    o=pd.read_csv(os.path.join(REPO,'data/model_outputs',f'PSO_{year}_6params_NYC_norm_28_PSO_15.csv'))
    o['cost']=pd.to_numeric(o['cost'],errors='coerce')
    if abs(medfit-EXP7[year])>0.0001: ok7=False; print('   MISMATCH',year,medfit,'vs',EXP7[year])
    print(f"  {year:<5d} {medfit:>20.4f} {1-o['cost'].mean():>14.4f}")
check('stability median fit matches report', ok7)

## Result

Seeding is exactly reproducible. Per-CBG point estimates are noisy (median CV ≈ 0.31–0.56), but the
**population-level conclusions are stable**: the four headline parameters (store area↑, chain loyalty↑,
POI count↑, distance↓) are recovered with the same sign by **10/10 seeds** in both periods, while the two
null effects (POI diversity, demographic similarity) waver — exactly as expected. Report population means
with the across-seed std as an uncertainty band.

In [ ]:
print('='*60); print('NOTEBOOK 16 VERIFICATION SUMMARY'); print('='*60)
nfail=sum(1 for _,ok,_ in PASS if not ok)
for name,ok,_ in PASS: print(('PASS ' if ok else 'FAIL ')+name)
print('-'*60); print(f'{len(PASS)-nfail}/{len(PASS)} checks passed')
assert nfail==0, f'{nfail} verification checks FAILED'